In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import import_ipynb
from shared_functions import DataframeLoader, Shared
base_dir = 'C:\\Users\\Administrador\\Desktop\\360\\Liquidations-Incentives-Jupyter'
sub_direct = 'archivos-fuente-116'
folder_bbraun_source = os.path.join(base_dir, 'source', sub_direct)
folder_bbraun_output = os.path.join(base_dir, 'output')
print(folder_bbraun_source)

In [ ]:
dataframe_loader = DataframeLoader(
    base_dir=base_dir,
    sub_direct=sub_direct
)
shared = Shared()
print(dataframe_loader.cargar_dataframes)

In [ ]:
(
        df_resultado_detallado_previo,
        df_resultado_liquidaciones_previo,
        df_empleados,
        df_empleados_inactivos,
        df_tipos_empleados,
        df_clusters_empleado,
        df_zonas_empleado,
        df_salarios_variables,
        df_parrillas,
        df_venta_recaudo_real,
        df_venta_recaudo_presupuesto,
        df_venta_por_zona,
        df_recaudo_por_zona,
        df_rentabilidad,
        df_venta_otras_companias,
        df_recaudo_otras_companias,
        df_venta_recaudo_kams,
        df_rentabilidad_kam,
        df_venta_servicios_mtto,
        df_objetivos_cualitativos,
        df_kpi_calidad,
        df_incentivos_por_empleado,
        df_area_calculo_sba,
        df_factores_liquidacion,
        df_cluster_plan_real_recaudo,
        df_zona_plan_real_venta_ch,
        df_rentabilidad_zona,
        df_resultados_variables_cualitativas,
        df_zonas_clusters_empleado,
        df_centros_costos,
        df_empleados_centros_costos,
        df_centros_costos_grupos_productos_divisiones,
        df_parrillas_tipos_calculos,
        df_area_calculo_sba_centros_costos,
        df_renal_ambulatorio,
        df_TRM,
        codigos_ve_vc,
        _dataframes_entrada,
        fecha_liquidacion,
        meses_incentivos,
        OUTPUT_FOLDER
    ) = dataframe_loader.cargar_dataframes(base_dir, folder_bbraun_source)

In [ ]:
maestra_resultados_por_empleado = pd.read_excel(
        io=os.path.join(folder_bbraun_output, 'maestra_resultados_por_empleado.xlsx'),
        dtype={
            'Contexto': str,
            'Fecha': 'datetime64[ns]',
            'Variable': str,
            'PorcentajeCumplimiento':float ,
            'Real': float,
            'Presupuesto': float,
            'CodigoEmpleado': str
        }
    )
maestra_resultados_por_empleado

In [ ]:
# maestra_resultados_por_empleado = pd.read_excel(io=os.path.join(folder_bbraun_output, 'maestra_resultados_por_empleado.xlsx'))
# maestra_resultados_por_empleado['CodigoEmpleado'] = maestra_resultados_por_empleado['CodigoEmpleado'].astype(str)
# maestra_resultados_por_empleado

In [ ]:
maestra_resultados_por_empleado_centro_costo = pd.read_excel(
        io=os.path.join(folder_bbraun_output, 'maestra_resultados_por_empleado_centro_costo.xlsx'),
        dtype={
            'Contexto': str,
            'Fecha': 'datetime64[ns]',
            'Variable': str,
            'PorcentajeCumplimiento':float ,
            'Real': float,
            'Presupuesto': float,
            'CodigoEmpleado': str,
            'CodigoCentroCosto': str
        }
    )
maestra_resultados_por_empleado_centro_costo.drop(columns=['Unnamed: 0'], inplace=True)
maestra_resultados_por_empleado_centro_costo

In [ ]:
maestra_resultados_por_empleado_centro_costo[maestra_resultados_por_empleado_centro_costo['Variable'] == 'RentabilidadUnidadNegocioCM2COGSSUB']

In [ ]:
maestra_resultados_por_empleado_centro_costo[
    (maestra_resultados_por_empleado_centro_costo['CodigoEmpleado'] == '1032450706') &
    (maestra_resultados_por_empleado_centro_costo['Fecha'] == '2023-01-01')
]

In [ ]:
# maestra_resultados_por_empleado_centro_costo = pd.read_excel(io=os.path.join(folder_bbraun_output, 'maestra_resultados_por_empleado_centro_costo.xlsx'))
# maestra_resultados_por_empleado_centro_costo

In [ ]:
resultado_final = pd.read_excel(io=os.path.join(folder_bbraun_output, 'resultado_final.xlsx'))
resultado_final

## Casos Liquidación Centro de Costo

In [ ]:
# Convertir las columnas a str
resultado_final['Código'] = resultado_final['Código'].astype(str)
df_empleados_centros_costos['CodigoEmpleado'] = df_empleados_centros_costos['CodigoEmpleado'].astype(str)

In [ ]:
resultado_final_liquidacion_centro_costo = resultado_final.rename(columns={'Código': 'CodigoEmpleado'}).merge(
    df_empleados_centros_costos,
    on=['CodigoEmpleado'],
    how='left'
).rename(columns={
    'Fecha': 'Fecha Proceso',
    'PorcentajeAsignacion': 'Porcentaje Aplicado'
})
resultado_final_liquidacion_centro_costo.fillna(0, inplace=True)
resultado_final_liquidacion_centro_costo

In [ ]:
resultado_final_liquidacion_centro_costo[resultado_final_liquidacion_centro_costo['CodigoEmpleado'] == '80220240']

In [ ]:
resultado_final_liquidacion_centro_costo.to_excel('resultado_final_liquidacion_centro_costo.xlsx')

In [ ]:
# resultado_final_liquidacion_centro_costo['CodigoEmpleado'] = resultado_final_liquidacion_centro_costo['CodigoEmpleado'].astype(str)

### Caso 1 - Liquidación Centro de Costo

In [ ]:
liquidacion_centro_costo_caso1 = resultado_final_liquidacion_centro_costo[
    (resultado_final_liquidacion_centro_costo['Valor liquidado'] < resultado_final_liquidacion_centro_costo['Pagado a la fecha']) &
    (resultado_final_liquidacion_centro_costo['¿Pago garantizado?'] == 'NO')
].copy()

# Asignar nuevos valores usando .loc para evitar SettingWithCopyWarning
liquidacion_centro_costo_caso1.loc[:, 'Consecutivo'] = 0.0
liquidacion_centro_costo_caso1.loc[:, 'TipoCalculo'] = 'F'

liquidacion_centro_costo_caso1

In [ ]:
liquidacion_centro_costo_caso1['Diferencia'] = liquidacion_centro_costo_caso1['Pagado a la fecha'] - liquidacion_centro_costo_caso1['Valor liquidado']
liquidacion_centro_costo_caso1 = liquidacion_centro_costo_caso1.rename(columns={'Diferencia': 'Valor Base Cálculo'})[
    ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'Valor Base Cálculo', 'CodigoCentroCosto', 'Porcentaje Aplicado']
].copy()
liquidacion_centro_costo_caso1['Caso'] = 1 
liquidacion_centro_costo_caso1.head()

In [ ]:
liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['CodigoEmpleado'] == '80220240']

## Caso 3 - Liquidación Centro de Costo

In [ ]:
resultado_final_liquidacion_centro_costo[resultado_final_liquidacion_centro_costo['CodigoEmpleado'] == '1010166177']

In [ ]:
liquidacion_centro_costo_garantizado = resultado_final_liquidacion_centro_costo[
    resultado_final_liquidacion_centro_costo['¿Pago garantizado?'] == 'SI'
].copy()
# liquidacion_centro_costo_garantizado = liquidacion_centro_costo_garantizado.rename(columns={'Valor liquidado': 'Valor Base Cálculo'})
liquidacion_centro_costo_garantizado['Valor Base Cálculo'] = liquidacion_centro_costo_garantizado.groupby(['CodigoEmpleado', 'CodigoCentroCosto'])[['Valor liquidado']].transform('sum')
liquidacion_centro_costo_garantizado['Total Meses Garantizado'] = liquidacion_centro_costo_garantizado.groupby(['CodigoEmpleado', 'CodigoCentroCosto'])[['Valor liquidado']].transform('count')
liquidacion_centro_costo_garantizado

In [ ]:
liquidacion_centro_costo_garantizado = liquidacion_centro_costo_garantizado[
    [
        'Fecha Proceso', 'CodigoEmpleado', 'Valor Base Cálculo', 'CodigoCentroCosto', 'Porcentaje Aplicado'
    ]
]
liquidacion_centro_costo_garantizado

In [ ]:
liquidacion_centro_costo_garantizado[liquidacion_centro_costo_garantizado['CodigoEmpleado'] == '32298388']

In [ ]:
# Agrupar por las columnas específicas y luego seleccionar el último registro de cada grupo
ultimos_registros_liquidacion_centro_costo_garantizado = liquidacion_centro_costo_garantizado.copy().fillna(0).groupby(['CodigoEmpleado', 'Valor Base Cálculo', 'CodigoCentroCosto', 'Porcentaje Aplicado']).apply(lambda x: x.nlargest(1, 'Fecha Proceso')).reset_index(drop=True)
ultimos_registros_liquidacion_centro_costo_garantizado.head()

In [ ]:
# Reiniciar el índice si lo deseas
ultimos_registros_liquidacion_centro_costo_garantizado[ultimos_registros_liquidacion_centro_costo_garantizado['CodigoEmpleado'] == '1032449509']

In [ ]:
# Asignar nuevos valores usando .loc para evitar SettingWithCopyWarning
ultimos_registros_liquidacion_centro_costo_garantizado.loc[:, 'Consecutivo'] = 0.0
ultimos_registros_liquidacion_centro_costo_garantizado.loc[:, 'TipoCalculo'] = 'F'

ultimos_registros_liquidacion_centro_costo_garantizado[
    (ultimos_registros_liquidacion_centro_costo_garantizado['CodigoEmpleado'] == '1014204375')
]

In [ ]:
liquidacion_centro_costo_caso3 = ultimos_registros_liquidacion_centro_costo_garantizado[
    ultimos_registros_liquidacion_centro_costo_garantizado['Fecha Proceso'] < fecha_liquidacion
]
liquidacion_centro_costo_caso3.head()

In [ ]:
liquidacion_centro_costo_caso3['Fecha Proceso'] = pd.to_datetime(fecha_liquidacion)
liquidacion_centro_costo_caso3['Caso'] = 3

In [ ]:
liquidacion_centro_costo_caso3[liquidacion_centro_costo_caso3['CodigoEmpleado'] == '1111117']

## Caso 4 - Liquidación Centro de Costo

In [ ]:
empleados_inactivos = df_empleados_inactivos[df_empleados_inactivos['FechaRetiro'].notnull()][['CodigoEmpleado', 'FechaRetiro']]
fecha_liquidacion_actual = datetime.strptime(fecha_liquidacion, '%Y-%m-%d')
año_liquidacion_actual = fecha_liquidacion_actual.year
print(año_liquidacion_actual)
empleados_inactivos

In [ ]:
empleados_con_fecha_retiro_ano_actual = empleados_inactivos[empleados_inactivos['FechaRetiro'].dt.year == año_liquidacion_actual]
empleados_con_fecha_retiro_ano_actual

In [ ]:
df_A = resultado_final_liquidacion_centro_costo.rename(columns={
    'Tipo Empleado': 'TipoEmpleado',
    'Área de cálculo': 'AreaCalculo',
    'Salario variable': 'SalarioVariable'
 #   'Valor liquidado': 'Valor Base Cálculo'
})[    [
    'Fecha Proceso', 'CodigoEmpleado', 'CodigoCentroCosto', 'Valor liquidado',
    'Porcentaje Aplicado']]
df_A

In [ ]:
df_A[df_A['CodigoEmpleado'] == '1098612757']

In [ ]:
df_A[df_A['CodigoEmpleado'] == '1032450706']

In [ ]:
maestra_resultados_por_empleado_centro_costo.head()

In [ ]:
len(maestra_resultados_por_empleado.copy())

### Caso 4 - Maestro Resultados Por Empleado

In [ ]:
maestra_resultados_por_empleado[
    (maestra_resultados_por_empleado['CodigoEmpleado'] == '1032450706') &
    (maestra_resultados_por_empleado['Fecha'] == '2023-09-01')
]

In [ ]:
maestra_resultados_por_empleado.info()

In [ ]:
df_B = maestra_resultados_por_empleado.copy().rename(columns={
    'Fecha': 'Fecha Proceso' #, 'Liquidado': 'Valor Base Cálculo'
}).merge(df_empleados,
    on=['CodigoEmpleado', 'TipoEmpleado', 'AreaCalculo', 'Nombre', 'Apellidos', 'FechaIngreso', 'SalarioVariable'],
    how='left',
    indicator=False
)[
        ['Fecha Proceso', 'CodigoEmpleado', 'FechaRetiro', 'AreaCalculo', 'Consecutivo','Variable', 'TipoEmpleado', 'Liquidado']
]
df_B = df_B.dropna(subset=['CodigoEmpleado']).drop_duplicates()
df_B

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & (df_B['Fecha Proceso'] == '2023-09-01')]

In [ ]:
df_B = df_B.merge(
    df_empleados_centros_costos,
    on=['CodigoEmpleado'],
    how='left',
    indicator=False
)
df_B = df_B.dropna(subset=['CodigoCentroCosto'])
df_B

In [ ]:
df_B[(df_B['CodigoEmpleado'] == '1032450706') & (df_B['Fecha Proceso'] == '2023-09-01')]

In [ ]:
df_parrillas_tipos_calculos

In [ ]:
df_B = df_B.merge( 
    right=df_parrillas_tipos_calculos.rename(columns={'NombreRubro': 'Variable'}), 
    on=['TipoEmpleado', 'AreaCalculo', 'Variable'],
    how='left' 
) 

df_B[[
    'Fecha Proceso', 'CodigoEmpleado', 'FechaRetiro', 'Variable', 'AreaCalculo', 'PorcentajeAsignacion', 'TipoEmpleado', 
    'CodigoRubro', 'TipoCalculo', 'CodigoCentroCosto', 'Liquidado' 
]]


In [ ]:
df_B[df_B['CodigoEmpleado'] == '5002899']

### Caso 4 - Maestro Resultados Por Empleado Centro de Costo

In [ ]:
maestra_resultados_por_empleado_centro_costo[
    (maestra_resultados_por_empleado_centro_costo['CodigoEmpleado'] == '1032450706') &
    (maestra_resultados_por_empleado_centro_costo['Fecha'] == '2023-09-01')
]

In [ ]:
maestra_resultados_por_empleado_centro_costo[maestra_resultados_por_empleado_centro_costo['CodigoEmpleado'] == '5002711']

In [ ]:
# df_B = maestra_resultados_por_empleado_centro_costo.copy().rename(columns={
#     'Fecha': 'Fecha Proceso' #, 'Liquidado': 'Valor Base Cálculo'
# }).merge(df_empleados,
#     on=['CodigoEmpleado', 'TipoEmpleado', 'AreaCalculo', 'Nombre', 'Apellidos', 'FechaIngreso', 'SalarioVariable'],
#     how='left',
#     indicator=False
# )[
#         ['Fecha Proceso', 'CodigoEmpleado', 'CodigoCentroCosto', 'FechaRetiro']
# ]
# df_B = df_B.dropna(subset=['CodigoEmpleado']).drop_duplicates()
# df_B

In [ ]:
df_B[df_B['CodigoEmpleado'] == '5002711']

In [ ]:
df_empleados_centros_costos[df_empleados_centros_costos['CodigoEmpleado'] == '5002711']

In [ ]:
# df_B = df_B.merge(
#     df_empleados_centros_costos,
#     on=['CodigoEmpleado', 'CodigoCentroCosto'],
#     how='left',
#     indicator=False
# )
# df_B = df_B.dropna(subset=['CodigoCentroCosto'])
# df_B

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & 
    (df_B['Fecha Proceso'] == '2023-09-01')
]

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & 
    (df_B['Fecha Proceso'] == '2023-09-01') &
    (df_B['TipoCalculo'] == 'F')
]

In [ ]:
3780000+6300000

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & 
    (df_B['Fecha Proceso'] == '2023-09-01') &
    (df_B['TipoCalculo'] == 'V')
]

In [ ]:
6048000+2340000+198000

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & 
    (df_B['Fecha Proceso'] == '2023-09-01') &
    (df_B['TipoCalculo'] == 'F')
]['Liquidado'].sum()

In [ ]:
df_B[
    (df_B['CodigoEmpleado'] == '1032450706') & 
    (df_B['Fecha Proceso'] == '2023-09-01') &
    (df_B['TipoCalculo'] == 'V')
]['Liquidado'].sum()

In [ ]:
# Haciendo un merge entre df_A y df_B con un indicador
merged_df = df_A.merge(df_B, on=[
    'CodigoEmpleado', 'Fecha Proceso', 'CodigoCentroCosto'], how='right', indicator=True
)
merged_df

In [ ]:
df_A[df_A['CodigoEmpleado'] == '1032450706']

In [ ]:
df_B[
        (df_B['CodigoEmpleado'] == '1032450706') &
        (df_B['Fecha Proceso'] == '2023-09-01')
    ]

In [ ]:
merged_df[
    (merged_df['CodigoEmpleado']=='1032450706') &
    (merged_df['Fecha Proceso']=='2023-09-01')
]

In [ ]:
# Filtrando aquellos registros que están en ambos dataframes (es decir, df_A, df_B)

# Registros que están dentro de resultado final liquidación por centro de costo y además aparecen en la maestra de resultados por empleado.

# liquidacion_centro_costo_caso4_both = merged_df[merged_df['_merge'] == 'both']
# resultado_final_liquidacion_centro_costo
# empleados_centro_costo_con_valor_liquidado

empleados_centro_costo_con_valor_liquidado = merged_df.dropna(subset=['Valor liquidado'])
empleados_centro_costo_con_valor_liquidado

In [ ]:
empleados_centro_costo_con_valor_liquidado[
    (empleados_centro_costo_con_valor_liquidado['CodigoEmpleado'] == '1032450706') &
    (empleados_centro_costo_con_valor_liquidado['Fecha Proceso'] == '2023-09-01')
]

In [ ]:
# # Agrupar por las columnas específicas y luego seleccionar el último registro de cada grupo
# ultimos_registros_empleados_centro_costo_con_valor_liquidado = empleados_centro_costo_con_valor_liquidado.groupby(['CodigoEmpleado', 'CodigoCentroCosto']).apply(lambda x: x.nlargest(1, 'Fecha Proceso'))

# # Reiniciar el índice si lo deseas
# ultimos_registros_empleados_centro_costo_con_valor_liquidado = ultimos_registros_empleados_centro_costo_con_valor_liquidado.reset_index(drop=True)
# ultimos_registros_empleados_centro_costo_con_valor_liquidado = ultimos_registros_empleados_centro_costo_con_valor_liquidado.drop(columns=['_merge'])

ultimos_registros_empleados_centro_costo_con_valor_liquidado = empleados_centro_costo_con_valor_liquidado.groupby(['CodigoEmpleado', 'TipoCalculo', 'CodigoCentroCosto', 'Consecutivo','Variable']).apply(
    lambda x: x.nlargest(1, 'Fecha Proceso')
).reset_index(drop=True).drop(columns=['_merge'])
ultimos_registros_empleados_centro_costo_con_valor_liquidado

In [ ]:
ultimos_registros_empleados_centro_costo_con_valor_liquidado[
    (ultimos_registros_empleados_centro_costo_con_valor_liquidado['CodigoEmpleado'] == '5002899') 
]

In [ ]:
481140.0+2430000.0+5852925.0+3137167.8

In [ ]:
1786860.0+7445250.0+3192523.2

In [ ]:
# Registros que fueron excluídos dentro de resultado final liquidación por centro de costo, pero sí aparecen en la maestra de resultados por empleado.

# Filtrando aquellos registros que solo están en el lado izquierdo (es decir, df_A)
# empleados_exluidos_centro_costo_con_valor_liquidado = merged_df[merged_df['_merge'] == 'right_only']
# empleados_exluidos_centro_costo_con_valor_liquidado = empleados_exluidos_centro_costo_con_valor_liquidado.drop(columns=['Valor liquidado', 'Fecha Proceso', '_merge', 'Porcentaje Aplicado'])

empleados_exluidos_centro_costo_con_valor_liquidado =  merged_df[merged_df['Valor liquidado'].isna()].drop(columns=[
    'Valor liquidado', 'Porcentaje Aplicado', 'Fecha Proceso', '_merge', 'FechaRetiro', 'CodigoRubro', 'TipoCalculo', 'Liquidado'
]).drop_duplicates()
empleados_exluidos_centro_costo_con_valor_liquidado

In [ ]:
empleados_exluidos_centro_costo_con_valor_liquidado[
    empleados_exluidos_centro_costo_con_valor_liquidado['CodigoEmpleado'] == '8520024'
]

In [ ]:
empleados_exluidos_centro_costo_con_valor_liquidado[
    empleados_exluidos_centro_costo_con_valor_liquidado['CodigoEmpleado'] == '1032450706'
]

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos = ultimos_registros_empleados_centro_costo_con_valor_liquidado.merge(
    empleados_exluidos_centro_costo_con_valor_liquidado.drop(
        columns=['AreaCalculo', 'TipoEmpleado', 'Variable' , 'Consecutivo']
    ).copy(),
    how='right',
    on=['CodigoEmpleado', 'CodigoCentroCosto', 'PorcentajeAsignacion']
).drop_duplicates()
resultado_ultima_liquidacion_empleados_retirados_o_inactivos

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos[
    resultado_ultima_liquidacion_empleados_retirados_o_inactivos['CodigoEmpleado'] == '5016583'
]

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos.info()

In [ ]:
empleados_con_fecha_retiro_ano_actual

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos = resultado_ultima_liquidacion_empleados_retirados_o_inactivos.rename(
        columns={'Fecha Proceso': 'Fecha Proceso Última Liquidacion'}
    ).merge(
    empleados_con_fecha_retiro_ano_actual,
    how='outer',
    on=['CodigoEmpleado', 'FechaRetiro']
).drop_duplicates()
resultado_ultima_liquidacion_empleados_retirados_o_inactivos['Fecha Proceso'] = pd.to_datetime(fecha_liquidacion)
resultado_ultima_liquidacion_empleados_retirados_o_inactivos

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos[
    resultado_ultima_liquidacion_empleados_retirados_o_inactivos['CodigoEmpleado'] == '8520024'
]

In [ ]:
    len(resultado_ultima_liquidacion_empleados_retirados_o_inactivos)

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos.to_excel('liquidacion_centro_costo_caso4_detallado.xlsx')

In [ ]:
# Eliminando la columna del indicador para limpiar el resultado
liquidacion_centro_costo_caso4 = resultado_ultima_liquidacion_empleados_retirados_o_inactivos.copy().rename(columns={'Liquidado': 'Valor Base Cálculo'})[
    [
    'Fecha Proceso', 'CodigoEmpleado', 'CodigoCentroCosto', 'Valor Base Cálculo',
    'Porcentaje Aplicado', 'TipoCalculo', 'Variable', 'Consecutivo', 'Fecha Proceso Última Liquidacion']
]
liquidacion_centro_costo_caso4['Fecha Proceso'] = liquidacion_centro_costo_caso4['Fecha Proceso Última Liquidacion']
liquidacion_centro_costo_caso4['Caso'] = 4
liquidacion_centro_costo_caso4.drop(columns=['Fecha Proceso Última Liquidacion', 'Variable'], inplace=True)
# liquidacion_centro_costo_caso4.loc[:, 'Consecutivo'] = 0.0
liquidacion_centro_costo_caso4.drop_duplicates(inplace=True)
liquidacion_centro_costo_caso4

In [ ]:
# liquidacion_centro_costo_caso4[liquidacion_centro_costo_caso4['TipoCalculo'] == 'F']

In [ ]:
# liquidacion_centro_costo_caso4[liquidacion_centro_costo_caso4['TipoCalculo'] == 'V']

In [ ]:
liquidacion_centro_costo_caso4[liquidacion_centro_costo_caso4['CodigoEmpleado'] == '8520024']

In [ ]:
liquidacion_centro_costo_caso4[liquidacion_centro_costo_caso4['CodigoEmpleado'] == '1032450706']

In [ ]:
df_empleados_inactivos['CodigoEmpleado'].unique()

In [ ]:
resultado_ultima_liquidacion_empleados_retirados_o_inactivos['CodigoEmpleado'].unique()

In [ ]:
empleados_con_fecha_retiro_ano_actual['CodigoEmpleado'].unique()

In [ ]:
cruzados = df_empleados_inactivos[['CodigoEmpleado']].merge(
    resultado_ultima_liquidacion_empleados_retirados_o_inactivos[['CodigoEmpleado']],
    how='outer',
    indicator=True
)
cruzados = cruzados.drop_duplicates()
cruzados

In [ ]:
cruzados[cruzados['_merge'] == 'left_only']

In [ ]:
cruzados[cruzados['_merge'] == 'right_only']

In [ ]:
cruzados[cruzados['_merge'] == 'both']

In [ ]:
liquidacion_centro_costo_caso4['CodigoEmpleado'].unique()

In [ ]:
liquidacion_centro_costo_caso4.to_excel('liquidacion_centro_costo_caso4.xlsx')

## Caso 5 - Liquidación Centro de Costo

In [ ]:
fecha_liquidacion

In [ ]:
liquidacion_centro_costo_caso5 = ultimos_registros_liquidacion_centro_costo_garantizado[
    ultimos_registros_liquidacion_centro_costo_garantizado['Fecha Proceso'] == fecha_liquidacion
]

liquidacion_centro_costo_caso5['Caso'] = 5

liquidacion_centro_costo_caso5

## Corrección Caso 1

In [ ]:
datos = {
    '1111117': 300000.0,
    '5032456': -300000.0
}

df_ajustes_liquidaciones = pd.DataFrame(list(datos.items()), columns=['CodigoEmpleado', 'AjusteValorLiquidado'])
df_ajustes_liquidaciones

In [ ]:
df_ajustes_liquidaciones.info()

In [ ]:
len(liquidacion_centro_costo_caso1)

In [ ]:
len(liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['Fecha Proceso'] < fecha_liquidacion])

In [ ]:
len(liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['Fecha Proceso'] == fecha_liquidacion])

In [ ]:
ultimos_liquidacion_centro_costo_caso1 = liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['Fecha Proceso'] == fecha_liquidacion]
ultimos_liquidacion_centro_costo_caso1

In [ ]:
_liquidacion_centro_costo_garantizado = pd.concat([
    liquidacion_centro_costo_caso3,
    liquidacion_centro_costo_caso5
])
_liquidacion_centro_costo_garantizado

In [ ]:
liquidacion_centro_costo_caso4

In [ ]:
ajuste_liquidacion_centro_costo_caso1 = ultimos_liquidacion_centro_costo_caso1.copy().drop(columns=['Caso']).merge(
    _liquidacion_centro_costo_garantizado.copy().drop(columns=['Caso']).rename(columns={'Valor Base Cálculo': 'Valor Garantizado'}),
    how='left'
).merge(
    df_ajustes_liquidaciones,
    how='left',
    on=['CodigoEmpleado']
)
ajuste_liquidacion_centro_costo_caso1.fillna(0, inplace=True)
ajuste_liquidacion_centro_costo_caso1.head()

In [ ]:
len(liquidacion_centro_costo_caso4)

In [ ]:
liquidacion_centro_costo_caso4[liquidacion_centro_costo_caso4['CodigoEmpleado'] == '5016583']

In [ ]:
len(ajuste_liquidacion_centro_costo_caso1)

In [ ]:
ajuste_liquidacion_centro_costo_caso1[ajuste_liquidacion_centro_costo_caso1['CodigoEmpleado'] == '5004523']

In [ ]:
liquidacion_centro_costo_merge_casos_1_4 = ajuste_liquidacion_centro_costo_caso1.copy().merge(
    liquidacion_centro_costo_caso4.copy().drop(columns=['Caso', 'Valor Base Cálculo']),
    how='outer',
    indicator=True
)
liquidacion_centro_costo_merge_casos_1_4.head()

In [ ]:
liquidacion_centro_costo_merge_casos_1_4[liquidacion_centro_costo_merge_casos_1_4['CodigoEmpleado'] == '5004523']

In [ ]:
len(liquidacion_centro_costo_merge_casos_1_4[
    liquidacion_centro_costo_merge_casos_1_4['_merge'] == 'left_only'
])

In [ ]:
excluir_empleados_inactivos_liquidacion_centro_costo_caso1 = liquidacion_centro_costo_merge_casos_1_4[
    liquidacion_centro_costo_merge_casos_1_4['_merge'] == 'left_only'
]
excluir_empleados_inactivos_liquidacion_centro_costo_caso1

In [ ]:
len(liquidacion_centro_costo_merge_casos_1_4[
    liquidacion_centro_costo_merge_casos_1_4['_merge'] == 'both'
])

In [ ]:
liquidacion_centro_costo_merge_casos_1_4[
    liquidacion_centro_costo_merge_casos_1_4['_merge'] == 'both'
]

In [ ]:
excluir_empleados_inactivos_liquidacion_centro_costo_caso1.loc[:, 'Valor Final'] = (
    excluir_empleados_inactivos_liquidacion_centro_costo_caso1['Valor Base Cálculo']
    - excluir_empleados_inactivos_liquidacion_centro_costo_caso1['Valor Garantizado']
    - excluir_empleados_inactivos_liquidacion_centro_costo_caso1['AjusteValorLiquidado']
).abs()
excluir_empleados_inactivos_liquidacion_centro_costo_caso1

In [ ]:
excluir_empleados_inactivos_liquidacion_centro_costo_caso1[excluir_empleados_inactivos_liquidacion_centro_costo_caso1['CodigoEmpleado'] == '5004523']

In [ ]:
liquidacion_centro_costo_caso1 = excluir_empleados_inactivos_liquidacion_centro_costo_caso1.copy().drop(
        columns=['Valor Base Cálculo']
    ).rename(
            columns={'Valor Final': 'Valor Base Cálculo'}
        )[
            ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'Valor Base Cálculo', 'CodigoCentroCosto', 'Porcentaje Aplicado']
         ].copy()
liquidacion_centro_costo_caso1['Caso'] = 1 
liquidacion_centro_costo_caso1.head()

In [ ]:
liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['CodigoEmpleado'] == '5004523']

In [ ]:
# liquidacion_centro_costo_caso1[liquidacion_centro_costo_caso1['Fecha Proceso'] == fecha_liquidacion].drop(
#         columns=['Caso']
#     ).rename(columns={'Valor Base Cálculo': 'Valor Total Liquidado'}).merge(
#             liquidacion_centro_costo_caso3.drop(columns=['Caso']).rename(columns={'Valor Base Cálculo': 'Valor Liquidado Garantizado'}),
#             how='left'
#         )

In [ ]:
liquidacion_centro_costo_caso5

## Concatenación Casos - Liquidación Centro de Costo

In [ ]:
liquidaciones_centro_costo_casos = pd.concat([
    liquidacion_centro_costo_caso1,
    liquidacion_centro_costo_caso3,
    liquidacion_centro_costo_caso4,
    liquidacion_centro_costo_caso5
], ignore_index=True)
liquidaciones_centro_costo_casos

In [ ]:
# liquidaciones_centro_costo_casos['Valor Calculado'] = liquidaciones_centro_costo_casos['Valor Base Cálculo'] * liquidaciones_centro_costo_casos['Porcentaje Aplicado']
# liquidaciones_centro_costo_casos.head()

In [ ]:
liquidaciones_centro_costo_casos.to_excel("liquidaciones_centro_costo_casos.xlsx")

## Se realizan las liquidaciones por Centros de Costos según el Tipo de Cálculo de las Variables

In [ ]:
maestra_resultados_por_empleado[
    (maestra_resultados_por_empleado['CodigoEmpleado'] == '5016583') &
    (maestra_resultados_por_empleado['Fecha'] == '2023-08-01')
]

In [ ]:
main = maestra_resultados_por_empleado.copy().merge(
    right=df_parrillas_tipos_calculos.rename(columns={'NombreRubro': 'Variable'}),
    on=['TipoEmpleado', 'AreaCalculo', 'Variable'],
    how='left'
)
main = main[['Contexto', 'Fecha', 'Variable', 'PorcentajeCumplimiento', 'CodigoEmpleado', 'Porcentaje', 'Consecutivo', 'TipoEmpleado', 'AreaCalculo', 'Liquidado','CodigoRubro', 'TipoCalculo']]
main

In [ ]:
main[
    (main['CodigoEmpleado'] == '5016583') &
    (main['Fecha'] == '2023-08-01')
]

### Tipo de Cálculo Fijo

In [ ]:
# final_calculo_fijo = pd.concat([
#     final_calculo_fijo,
#     liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'F']
# ])

In [ ]:
calculo_fijo = main[main['TipoCalculo']== 'F']
calculo_fijo

In [ ]:
calculo_fijo = calculo_fijo.groupby(['Fecha', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo'], as_index=False)[['Liquidado']].apply(sum)
calculo_fijo

In [ ]:
calculo_fijo[calculo_fijo['CodigoEmpleado'] == '5005463']

In [ ]:
liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'F']

In [ ]:
calculo_fijo.to_excel('calculo_fijo.xlsx')

In [ ]:
final_calculo_fijo = calculo_fijo.merge(
    df_empleados_centros_costos,
    on=['CodigoEmpleado'],
    how='left'
).rename(columns={
    'Fecha': 'Fecha Proceso',
    'Liquidado':'Valor Base Cálculo',
    'PorcentajeAsignacion': 'Porcentaje Aplicado'
})
final_calculo_fijo['Caso'] = 0
final_calculo_fijo

In [ ]:
 liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'F']

In [ ]:
#  liquidaciones_centro_costo_casos[
#      (liquidaciones_centro_costo_casos['TipoCalculo'] == 'F') &
#      (liquidaciones_centro_costo_casos['Fecha Proceso'] < '2023-12-01')
#  ]['Fecha Proceso'] = pd.to_datetime(fecha_liquidacion)

In [ ]:
liquidaciones_centro_costo_casos.loc[
    (liquidaciones_centro_costo_casos['TipoCalculo'] == 'F') &
    (liquidaciones_centro_costo_casos['Fecha Proceso'] < fecha_liquidacion),
    'Fecha Proceso'
] = pd.to_datetime(fecha_liquidacion)

In [ ]:
 liquidaciones_centro_costo_casos[
     (liquidaciones_centro_costo_casos['TipoCalculo'] == 'V') &
     (liquidaciones_centro_costo_casos['CodigoEmpleado'] == '5002899')
 ]

In [ ]:
# liquidacion_centro_costo_caso4['Fecha Proceso'] = pd.to_datetime(fecha_liquidacion)


In [ ]:
final_calculo_fijo = pd.concat([
    final_calculo_fijo,
    liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'F']
])
final_calculo_fijo

In [ ]:
final_calculo_fijo['Valor Calculado'] = final_calculo_fijo['Valor Base Cálculo'] * final_calculo_fijo['Porcentaje Aplicado']
final_calculo_fijo

In [ ]:
final_calculo_fijo[
    (final_calculo_fijo['CodigoEmpleado'] == '5016583') &
     (final_calculo_fijo['Fecha Proceso'] == '2023-12-01')
]   

In [ ]:
 liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'F'].head()

In [ ]:
final_calculo_fijo.to_excel('final_calculo_fijo.xlsx')

In [ ]:
consolidado_final_calculo_fijo = final_calculo_fijo.copy()
consolidado_final_calculo_fijo[
    (consolidado_final_calculo_fijo['Fecha Proceso'] == '2023-02-01') &
    (consolidado_final_calculo_fijo['CodigoEmpleado'] == '1111117')
]

In [ ]:
consolidado_final_calculo_fijo['Total Repetidos'] = consolidado_final_calculo_fijo.groupby(
    ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'CodigoCentroCosto', 'Porcentaje Aplicado']
)['Valor Calculado'].transform('count')

In [ ]:
consolidado_final_calculo_fijo['Valor Calculado'] = consolidado_final_calculo_fijo.groupby(
    ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'CodigoCentroCosto', 'Porcentaje Aplicado']
)['Valor Calculado'].transform('sum')

In [ ]:
consolidado_final_calculo_fijo

In [ ]:
consolidado_final_calculo_fijo.to_excel('consolidado_final_calculo_fijo.xlsx')

In [ ]:
consolidado_final_calculo_fijo['Valor Base Cálculo'] = consolidado_final_calculo_fijo.copy().groupby(
    ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'CodigoCentroCosto', 'Porcentaje Aplicado']
)['Valor Base Cálculo'].transform('sum')

In [ ]:
# Eliminar las columnas 'Caso' y 'Total Repetidos'
consolidado_final_calculo_fijo_sin_repetidos = consolidado_final_calculo_fijo.copy().drop(columns=['Total Repetidos'])

# Eliminar registros duplicados sin tener en cuenta las columnas 'Caso' y 'Total Repetidos'
consolidado_final_calculo_fijo_sin_repetidos = consolidado_final_calculo_fijo_sin_repetidos.drop_duplicates(
    subset=['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'Valor Base Cálculo', 'CodigoCentroCosto', 'Porcentaje Aplicado', 'Valor Calculado'],
    keep='first'
)
consolidado_final_calculo_fijo_sin_repetidos

In [ ]:
consolidado_final_calculo_fijo_sin_repetidos[consolidado_final_calculo_fijo_sin_repetidos['CodigoEmpleado'] == '1098612757']

In [ ]:
consolidado_final_calculo_fijo_sin_repetidos.to_excel('consolidado_final_calculo_fijo_sin_repetidos.xlsx')

### Tipo de Cálculo Variable

In [ ]:
main.head()

In [ ]:
calculo_variable = main[main['TipoCalculo']== 'V']
calculo_variable['Caso'] = 0
calculo_variable.head()

In [ ]:
calculo_variable[
    (calculo_variable['CodigoEmpleado'] == '5002899') &
    (calculo_variable['Fecha'] == '2023-04-01')
]

In [ ]:
# maestra_resultados_por_empleado_centro_costo[
#     (maestra_resultados_por_empleado_centro_costo['CodigoEmpleado'] == '1032450706') &
#      (maestra_resultados_por_empleado_centro_costo['Fecha'] == '2023-09-01')
# ]   

In [ ]:
calculo_variable[
    (calculo_variable['TipoCalculo'] == 'V') &
    (calculo_variable['CodigoEmpleado'] == '5016583') &
    (calculo_variable['Fecha'] == '2023-09-01')
]

In [ ]:
liquidaciones_centro_costo_casos[
    (liquidaciones_centro_costo_casos['TipoCalculo'] == 'V') &
    (liquidaciones_centro_costo_casos['CodigoEmpleado'] == '5002899')
]

In [ ]:
_calculo_variable_centro_costo_casos = calculo_variable.drop(columns=['Porcentaje', 'Caso', 'PorcentajeCumplimiento']).merge(
    liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'V'].rename(
    columns={'Fecha Proceso':'Fecha'}).drop(columns=['Consecutivo', 'Porcentaje Aplicado', 'Valor Base Cálculo']),
    how='right',
    on=['CodigoEmpleado', 'Fecha', 'TipoCalculo']
).drop_duplicates()

_calculo_variable_centro_costo_casos['Fecha'] = pd.to_datetime(fecha_liquidacion)
_calculo_variable_centro_costo_casos

In [ ]:
_calculo_variable_centro_costo_casos[_calculo_variable_centro_costo_casos['CodigoEmpleado'] == '5002899']

In [ ]:
_final_calculo_variable = pd.concat([
    calculo_variable.drop(columns=['Porcentaje']),
    _calculo_variable_centro_costo_casos
]).drop(columns=['PorcentajeCumplimiento', 'CodigoCentroCosto']).drop_duplicates()
_final_calculo_variable

In [ ]:
calculo_variable[
    (calculo_variable['TipoCalculo'] == 'V') & 
    (calculo_variable['CodigoEmpleado'] == '1032450706') &
     (calculo_variable['Fecha'] == '2023-09-01')
]

In [ ]:
liquidaciones_centro_costo_casos[
    (liquidaciones_centro_costo_casos['TipoCalculo'] == 'F') & 
    (liquidaciones_centro_costo_casos['CodigoEmpleado'] == '8520024')
]

In [ ]:
_final_calculo_variable[
    (_final_calculo_variable['CodigoEmpleado'] == '5002899') &
     (_final_calculo_variable['Fecha'] == '2023-12-01')
]   

In [ ]:
maestra_resultados_por_empleado_centro_costo[
    (maestra_resultados_por_empleado_centro_costo['CodigoEmpleado'] == '5002899')
]   

In [ ]:
final_calculo_variable = _final_calculo_variable.merge(
    right=maestra_resultados_por_empleado_centro_costo.rename(columns={'NombreRubro': 'Variable'}),
    on=['Fecha', 'Contexto', 'CodigoEmpleado', 'Variable', 'Consecutivo','TipoEmpleado', 'AreaCalculo'],
    how='left'
)[['Fecha', 'Contexto', 'CodigoEmpleado', 'TipoEmpleado','TipoCalculo', 'GrupoProducto', 'Division', 'Consecutivo', 'CodigoCentroCosto', 'Real','Variable', 'Liquidado', 'Caso']]
final_calculo_variable

In [ ]:
final_calculo_variable[
    (final_calculo_variable['CodigoEmpleado'] == '5002899') &
     (final_calculo_variable['Fecha'] == '2023-12-01')
]   

In [ ]:
final_calculo_variable.info()

In [ ]:
final_calculo_variable[
    (final_calculo_variable['CodigoEmpleado'] == '1032450706') &
     (final_calculo_variable['Fecha'] == '2023-09-01')
]

In [ ]:
final_calculo_variable = final_calculo_variable.groupby(['Fecha', 'CodigoEmpleado', 'TipoCalculo', 'CodigoCentroCosto', 'Consecutivo', 'Liquidado', 'Caso'], as_index=False)[['Real']].apply(sum)

final_calculo_variable.head()

In [ ]:
final_calculo_variable['RealTotal'] = final_calculo_variable.groupby(
    ['Fecha', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'Caso'], as_index=False
)['Real'].transform(pd.Series.sum)
final_calculo_variable

In [ ]:
final_calculo_variable['Valor Base Cálculo'] = final_calculo_variable.groupby(
    ['Fecha', 'CodigoEmpleado', 'TipoCalculo', 'CodigoCentroCosto', 'Consecutivo', 'Caso'], as_index=False
)['Liquidado'].transform(pd.Series.sum)
final_calculo_variable

In [ ]:
final_calculo_variable = final_calculo_variable.groupby(['Fecha', 'CodigoEmpleado', 'TipoCalculo', 'Caso', 'Consecutivo', 'CodigoCentroCosto', 'RealTotal', 'Valor Base Cálculo'], as_index=False)[['Real']].apply(sum)


In [ ]:
final_calculo_variable['Porcentaje Aplicado'] = final_calculo_variable['Real']/final_calculo_variable['RealTotal']
final_calculo_variable = final_calculo_variable.rename(columns={
    'Fecha': 'Fecha Proceso'
})
final_calculo_variable['Valor Calculado'] = final_calculo_variable['Valor Base Cálculo'] * final_calculo_variable['Porcentaje Aplicado']
final_calculo_variable

In [ ]:
final_calculo_variable = final_calculo_variable[
    ['Fecha Proceso', 'CodigoEmpleado', 'TipoCalculo', 'Consecutivo', 'CodigoCentroCosto', 'Valor Base Cálculo', 'Porcentaje Aplicado', 'Valor Calculado', 'Caso']
]
final_calculo_variable

In [ ]:
# final_calculo_variable = pd.concat([
#     final_calculo_variable,
#     liquidaciones_centro_costo_casos[liquidaciones_centro_costo_casos['TipoCalculo'] == 'V']
# ])
# final_calculo_variable

In [ ]:
final_calculo_variable[
    (final_calculo_variable['CodigoEmpleado'] == '5002899') &
     (final_calculo_variable['Fecha Proceso'] == '2023-12-01')
]

In [ ]:
final_calculo_variable[
    (final_calculo_variable['CodigoEmpleado'] == '1032450706') &
     (final_calculo_variable['Fecha Proceso'] == '2023-12-01')
]

In [ ]:
final_calculo_variable.to_excel('final_calculo_variable.xlsx')

### Se unifican Tipo Cálculo Fijo y Variable

In [ ]:
final_calculo_fijo.head()

In [ ]:
final_calculo_fijo[final_calculo_fijo['CodigoEmpleado'] == '1098612757']

In [ ]:
len(final_calculo_fijo[final_calculo_fijo['CodigoEmpleado'] == '1098612757'])

In [ ]:
consolidado_final_calculo_fijo_sin_repetidos[consolidado_final_calculo_fijo_sin_repetidos['CodigoEmpleado'] == '1098612757']

In [ ]:
len(consolidado_final_calculo_fijo_sin_repetidos[consolidado_final_calculo_fijo_sin_repetidos['CodigoEmpleado'] == '1098612757'])

In [ ]:
final_calculo_variable.head()

In [ ]:
final_calculo_fijo[
    (final_calculo_fijo['CodigoEmpleado'] == '1032450706') &
     (final_calculo_fijo['Fecha Proceso'] == '2023-09-01')
]

In [ ]:
final_calculo_variable[
    (final_calculo_variable['CodigoEmpleado'] == '1032450706') &
     (final_calculo_variable['Fecha Proceso'] == '2023-09-01')
]

In [ ]:
resultado_liquidaciones_centro_costo = pd.concat(
    [ final_calculo_fijo,
      final_calculo_variable,
#      consolidado_final_calculo_fijo_sin_repetidos
    ]
)
resultado_liquidaciones_centro_costo[
    (resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '8520024') &
     (resultado_liquidaciones_centro_costo['Fecha Proceso'] == '2023-09-01')
]

In [ ]:
resultado_liquidaciones_centro_costo[
    (resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '8520024') &
     (resultado_liquidaciones_centro_costo['Fecha Proceso'] == '2023-09-01') &
    (resultado_liquidaciones_centro_costo['TipoCalculo'] == 'F')
]['Valor Calculado'].sum()

In [ ]:
6048000.0+2340000.0+198000.0

In [ ]:
len(resultado_liquidaciones_centro_costo[resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '1098612757'])

In [ ]:
resultado_liquidaciones_centro_costo[
    (resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '5002899') &
    (resultado_liquidaciones_centro_costo['Fecha Proceso'] == '2023-12-01')
]

In [ ]:
nombre_archivo_salida = "resultado_liquidaciones_centro_costo (" + fecha_liquidacion + ").xlsx"
nombre_archivo_salida

In [ ]:
resultado_liquidaciones_centro_costo.to_excel(nombre_archivo_salida)

In [ ]:
resultado_liquidaciones_centro_costo[
    (resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '8520024') &
    (resultado_liquidaciones_centro_costo['Fecha Proceso'] == '2023-12-01')
]

In [ ]:
resultado_liquidaciones_centro_costo[resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '5016692']

In [ ]:
final_calculo_variable[final_calculo_variable['CodigoEmpleado'] == '1098612757']

In [ ]:
resultado_liquidaciones_centro_costo[resultado_liquidaciones_centro_costo['CodigoEmpleado'] == '5004293']